# Mental Health AI Chatbot — LLM Server (Google Colab)

This notebook runs the **Counselor Llama 3 (Q4)** model on Colab GPU and exposes it as an API.

### Setup Steps:
1. **Runtime → Change runtime type → GPU (T4)**
2. Run **Cell 1** to install dependencies
3. Upload `Counselor_Llama3_Q4.gguf` to your **Google Drive** first, then run **Cell 2** to mount & copy it
4. Enter your **ngrok auth token** in Cell 3
5. Run **Cell 4** to start the LLM server
6. Run **Cell 5** to get the public URL
7. The Flask backend will use this URL automatically (set `LLM_API_URL` env var)
8. Run **Cell 6** to keep alive

In [ ]:
# ============================================
# CELL 1: Install Dependencies
# ============================================
!pip install -q llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu124
!pip install -q flask pyngrok
print("✅ All dependencies installed!")

## Load Model from Google Drive

Upload `Counselor_Llama3_Q4.gguf` (~4.58 GB) to your **Google Drive** first (use drive.google.com).  
This cell mounts your Drive and copies the model to the Colab runtime.

In [ ]:
# ============================================
# CELL 2: Load Model from Google Drive
# ============================================
import os
import shutil
from google.colab import drive

MODEL_FILE = 'Counselor_Llama3_Q4.gguf'

# ⚠️ EDIT THIS if your .gguf file is in a subfolder inside Drive
DRIVE_PATH = '/content/drive/MyDrive/Counselor_Llama3_Q4.gguf'

if os.path.exists(MODEL_FILE):
    size_gb = os.path.getsize(MODEL_FILE) / (1024**3)
    print(f"✅ Model already present: {MODEL_FILE} ({size_gb:.2f} GB)")
else:
    print("📂 Mounting Google Drive...")

    drive.mount('/content/drive')        print(f"✅ Model copied: {MODEL_FILE}")

        shutil.copy(DRIVE_PATH, MODEL_FILE)

    if not os.path.exists(DRIVE_PATH):        print(f"📦 Copying model ({size_gb:.2f} GB) to Colab runtime...")

        print(f"❌ Model not found at: {DRIVE_PATH}")        size_gb = os.path.getsize(DRIVE_PATH) / (1024**3)

        print("   Upload Counselor_Llama3_Q4.gguf to your Google Drive root")    else:
        print("   and re-run this cell. Or edit DRIVE_PATH above.")

In [ ]:
# ============================================
# CELL 3: Setup ngrok
# ============================================
from pyngrok import ngrok

# ⚠️ GET YOUR FREE TOKEN FROM: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"  # ← REPLACE THIS!

if NGROK_TOKEN == "YOUR_NGROK_AUTH_TOKEN_HERE":
    print("❌ Please replace NGROK_TOKEN with your actual token!")
    print("   Get it from: https://dashboard.ngrok.com/get-started/your-authtoken")
else:
    ngrok.set_auth_token(NGROK_TOKEN)
    print("✅ ngrok authenticated!")

In [ ]:
# ============================================
# CELL 4: Load Model & Start LLM Server
# ============================================
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

from llama_cpp import Llama

print("\n⏳ Loading Counselor Llama 3 model (this takes 1-2 minutes)...")
llm = Llama(
    model_path="Counselor_Llama3_Q4.gguf",
    n_ctx=2048,
    n_gpu_layers=-1,  # Offload all layers to GPU
    verbose=False
)
print("✅ Model loaded on GPU!")

# Start Flask API server
from flask import Flask, request, jsonify
import threading

app = Flask(__name__)

@app.route('/health', methods=['GET'])
def health():
    return jsonify({"status": "ok", "model": "Counselor_Llama3_Q4", "gpu": torch.cuda.is_available()})

@app.route('/generate', methods=['POST'])
def generate():
    data = request.get_json()
    prompt = data.get('prompt', '')
    if not prompt:
        return jsonify({"error": "Missing 'prompt' field"}), 400

    max_tokens = data.get('max_tokens', 300)
    temperature = data.get('temperature', 0.7)

    output = llm(
        prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        stop=["User:", "Human:", "<|eot_id|>"],
        echo=False
    )

    response_text = output['choices'][0]['text'].strip()
    return jsonify({"response": response_text})

# Run in background thread
threading.Thread(target=lambda: app.run(host='0.0.0.0', port=5000, debug=False), daemon=True).start()
import time; time.sleep(2)
print("✅ LLM API server running on port 5000!")

In [ ]:
# ============================================
# CELL 5: Get Public URL
# ============================================
public_url = ngrok.connect(5000, bind_tls=True)

print()
print("=" * 70)
print("🎉 LLM SERVER IS LIVE!")
print("=" * 70)
print(f"\n📍 LLM API URL (copy this):")
print(f"   {public_url}")
print(f"\n📍 Health Check:")
print(f"   {public_url}/health")
print(f"\n📍 Generate Endpoint:")
print(f"   POST {public_url}/generate")
print("\n" + "=" * 70)
print("\n📋 NEXT STEPS:")
print("   On your laptop, set the environment variable before running app.py:")
print(f'   $env:LLM_API_URL="{public_url}"')
print(f"   python app.py")
print("\n⏱️  Session expires in ~12 hours")
print("=" * 70)

In [ ]:
# ============================================
# CELL 6: Keep Server Running
# ============================================
import time

print("🔄 LLM Server is running...")
print("💡 Keep this cell running to keep the server alive")
print("⏹️  Click the stop button to shutdown\n")

counter = 0
try:
    while True:
        time.sleep(60)
        counter += 1
        print(f"⏰ Running for {counter} minutes...", end="\r")
except KeyboardInterrupt:
    print("\n✅ Server stopped!")